In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
# 오픈 AI 써도 되는데 키값이 필요함

In [2]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model = 'gemma2:2b',
    temperature = 0.5, # 생성되는 텍스트의 다양성을 조절하는 매개변수
    request_timeout = 120.0 # 요청이 타임아웃되기까지의 시간(초)_이 시간이 되면 멈춰라
)

embed_model = OllamaEmbedding(
    model_name = 'nomic-embed-text'
)

In [3]:
# 데이터 로드
documents = SimpleDirectoryReader('../Data/pdf_sample1/').load_data()

In [4]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(
    documents, # 데이터 
    embed_model = embed_model,
    show_progress = True # 진행 상황 표시 여부
    )

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 41/41 [00:03<00:00, 11.82it/s]


In [6]:
# 벡터 DB 관련 라마 인덱스 패키지
from llama_index.core import StorageContext, load_index_from_storage
persist_dir = './saved_index'

In [7]:
# 저장된 인덱스 로드
index.storage_context.persist(persist_dir)

In [8]:
# 저장된 인덱스 로드 위치 정의
storage_context = StorageContext.from_defaults(persist_dir = persist_dir)

In [9]:
# 인덱스 로드
loaded_index = load_index_from_storage(storage_context, embed_model = embed_model)

2026-05-19 11:14:01,853 - INFO - Loading all indices.


In [ ]:
# 쿼리 엔진
loaded_query_engine = loaded_index.as_query_engine(llm = llm)
# 저장된거 불러서 만든 새로운 엔진

2026-05-19 11:14:21,967 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [ ]:
# 쿼리 실행
query = "이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요."
response = loaded_query_engine.query(query)
# loaded_query_engine : 저장된 인덱스에서 만든 쿼리 엔진
# 응답 출력
print("\n질문: ", query)
print("답변: ", response)
# 더 자세한 답변을 원하면 파라메터 중 temperature 값을 높이면 된다 

2026-05-19 11:16:24,827 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:16:29,328 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문:  이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요.
답변:  본 문서에서는 AI 규제 방안에 대한 다양한 의견이 나왔습니다. 특히, 초당파와 민주당 사이에서 AI 규제의 효과 및 부작용에 대해 논쟁이 이루어졌다고 언급했습니다.  



#### 메모리 부분

In [18]:
# 쿼리 엔진
query_engine = index.as_query_engine(llm = llm)

In [19]:
# 쿼리 실행
query = "이 논문에서 제안하는 모델의 장점은 뭐야? 한글로 답변 하세요"
response = query_engine.query(query)

# 응답 출력
print("\n질문: ", query)
print("답변: ", response)

2026-05-19 11:18:54,350 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:18:56,147 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문:  이 논문에서 제안하는 모델의 장점은 뭐야? 한글로 답변 하세요
답변:  이 논문에서는 미국 AI 정책과 전략 현황을 중심으로 AI 지배력 강화와 초강대국 유지를 위한 변화 방향에 대해 다루고 있습니다.  



In [9]:
# temperature 를 수정하면 다양성이 높아짐. 낮추면 더 일관된 답변이 나옴. 0.5 정도가 적당한듯